In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt

In [2]:
orders = pd.read_csv('../data/raw/JD_order_data.csv')
delivery = pd.read_csv('../data/raw/JD_delivery_data.csv')
inventory = pd.read_csv('../data/raw/JD_inventory_data.csv')
skus = pd.read_csv('../data/raw/JD_sku_data.csv')

In [3]:
# 2) Build order-level mart

def to_dt(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce")

def build_order_mart(orders: pd.DataFrame,
                     delivery: pd.DataFrame,
                     inventory: pd.DataFrame,
                     skus: pd.DataFrame | None = None) -> pd.DataFrame:
    o = orders.copy()

    # Parse timestamps
    o["order_time_dt"] = to_dt(o["order_time"])
    o["order_date_dt"] = pd.to_datetime(o["order_date"], errors="coerce")

    # Cross-warehouse fulfillment
    o["cross_fulfill"] = (o["dc_ori"].astype("Int64") != o["dc_des"].astype("Int64")).astype(int)

    # Delivery aggregation (package-level -> order-level)
    d = delivery.copy()
    d["ship_out_dt"] = to_dt(d["ship_out_time"])
    d["arr_station_dt"] = to_dt(d["arr_station_time"])
    d["arr_dt"] = to_dt(d["arr_time"])

    # If multiple packages per order, use max(arr_dt) as final delivery time (conservative)
    d_agg = (d.groupby("order_ID", as_index=False)
               .agg(ship_out_dt=("ship_out_dt", "min"),
                    arr_station_dt=("arr_station_dt", "min"),
                    arr_dt=("arr_dt", "max")))

    mart = o.merge(d_agg, on="order_ID", how="left")

    # Lead time: order placement -> delivered
    mart["lead_time_hours"] = (mart["arr_dt"] - mart["order_time_dt"]).dt.total_seconds() / 3600.0
    mart["cross_filling_hours"] = (mart["arr_station_dt"] - mart["ship_out_dt"]).dt.total_seconds() / 3600.0

    # Promise: in days; some rows have '-' so coerce to NaN
    mart["promise_days"] = pd.to_numeric(mart["promise"], errors="coerce")
    mart["promise_hours"] = mart["promise_days"] * 24.0

    mart["late_delivery"] = np.where(
        mart["lead_time_hours"].notna() & mart["promise_hours"].notna(),
        (mart["lead_time_hours"] > mart["promise_hours"]).astype(int),
        np.nan
    )

    # Inventory availability at destination DC (binary proxy):
    # inventory table only records (dc_ID, sku_ID, date) IF available at end-of-day
    inv = inventory.copy()
    inv["date_dt"] = pd.to_datetime(inv["date"], errors="coerce")
    inv["inv_available"] = 1

    inv_key = inv[["dc_ID", "sku_ID", "date_dt", "inv_available"]].drop_duplicates()

    mart = mart.merge(
        inv_key,
        left_on=["dc_des", "sku_ID", "order_date_dt"],
        right_on=["dc_ID", "sku_ID", "date_dt"],
        how="left"
    )
    mart.rename(columns={"inv_available": "inv_available_at_dc_des"}, inplace=True)
    mart["inv_available_at_dc_des"] = mart["inv_available_at_dc_des"].fillna(0).astype(int)

    # Optional: attach SKU info (1P vs 3P, attributes)
    if skus is not None and "sku_ID" in skus.columns:
        sku_cols = [c for c in ["sku_ID", "type", "attribute1", "attribute2", "brand_ID"] if c in skus.columns]
        mart = mart.merge(skus[sku_cols].rename(columns={"type": "sku_type"}), on="sku_ID", how="left")

    return mart

mart = build_order_mart(orders, delivery, inventory, skus)

# mart = mart[mart["type"] == 1].copy()

drop_cols = [
        "order_date", "order_time", "promise",
        "original_unit_price", "final_unit_price",
        "direct_discount_per_unit", "quantity_discount_per_unit",
        "bundle_discount_per_unit", "coupon_discount_per_unit",
        "gift_item", "order_date_dt", "sku_type", "type"
            ]

mart = mart.drop(columns=[c for c in drop_cols if c in mart.columns])

mart.to_csv('../data/processed/JD_order_mart.csv', index=False)


In [4]:
mart

,order_ID,user_ID,sku_ID,quantity,dc_ori,dc_des,order_time_dt,cross_fulfill,ship_out_dt,arr_station_dt,...,cross_filling_hours,promise_days,promise_hours,late_delivery,dc_ID,date_dt,inv_available_at_dc_des,attribute1,attribute2,brand_ID
0,d0cf5cc6db,0abe9ef2ce,581d5b54c1,1,4,28,2018-03-01 17:14:25,1,NaT,NaT,...,NaN,NaN,NaN,NaN,NaN,NaT,0,4.0,100.0,198cec62a1
1,7444318d01,33a9e56257,067b673f2b,1,28,28,2018-03-01 11:10:40,0,2018-03-01 13:00:00,2018-03-02 08:00:00,...,19.0,2.0,48.0,0.0,NaN,NaT,0,3.0,60.0,9b0d3a5fc6
2,f973b01694,4ea3cf408f,623d0a582a,1,28,28,2018-03-01 09:13:26,0,2018-03-01 14:00:00,2018-03-02 09:00:00,...,19.0,2.0,48.0,0.0,28.0,2018-03-01,1,NaN,NaN,NaN
3,8c1cec8d4b,b87cb736cb,fc5289b139,1,4,28,2018-03-01 21:29:50,1,2018-03-02 09:00:00,2018-03-03 08:00:00,...,23.0,2.0,48.0,1.0,28.0,2018-03-01,1,NaN,NaN,NaN
4,d43a33c38a,4829223b6f,623d0a582a,1,3,16,2018-03-01 19:13:37,1,2018-03-01 20:00:00,2018-03-02 07:00:00,...,11.0,1.0,24.0,0.0,NaN,NaT,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549984,3ad06b9fbe,a27b3ed4d4,a9109972d1,1,2,2,2018-03-31 01:22:47,0,NaT,NaT,...,NaN,NaN,NaN,NaN,NaN,NaT,0,-,-,060cb5709b
549985,c9d77a7ed0,18f92434cd,7f53769d3f,1,59,2,2018-03-31 08:55:57,1,2018-03-31 11:00:00,2018-04-02 07:00:00,...,44.0,3.0,72.0,0.0,NaN,NaT,0,4.0,100.0,0b0f75e8d5
549986,b9ad79338f,b5caf8a580,8dc4a01dec,1,2,2,2018-03-31 13:31:01,0,2018-03-31 14:00:00,2018-04-01 07:00:00,...,17.0,2.0,48.0,0.0,2.0,2018-03-31,1,3.0,100.0,43999af013
549987,be3a9414b1,20ba6655f3,2dd6b818ec,1,4,28,2018-03-31 12:51:18,1,NaT,NaT,...,NaN,NaN,NaN,NaN,NaN,NaT,0,3.0,100.0,5ab8ea8556


In [5]:
# 3) Core summary metrics

def summarize(mart: pd.DataFrame):
    out = {}

    out["cross_fulfillment_rate"] = float(mart["cross_fulfill"].mean())

    lt = mart.loc[mart["lead_time_hours"].notna(), "lead_time_hours"]
    out["lead_time_mean_hours"] = float(lt.mean()) if len(lt) else np.nan
    out["lead_time_median_hours"] = float(lt.median()) if len(lt) else np.nan

    out["lead_time_by_fulfillment"] = (
        mart[mart["lead_time_hours"].notna()]
        .groupby("cross_fulfill")["lead_time_hours"]
        .agg(["count", "mean", "median", "std"])
        .rename(index={0: "local", 1: "cross"})
    )

    late_defined = mart["late_delivery"].notna()
    out["late_rate_by_fulfillment"] = (
        mart[late_defined]
        .groupby("cross_fulfill")["late_delivery"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "late_rate"})
        .rename(index={0: "local", 1: "cross"})
        if late_defined.any() else pd.DataFrame()
    )

    # Delay penalty: cross - local
    if "local" in out["lead_time_by_fulfillment"].index and "cross" in out["lead_time_by_fulfillment"].index:
        out["delay_penalty_mean_hours"] = float(
            out["lead_time_by_fulfillment"].loc["cross", "mean"]
            - out["lead_time_by_fulfillment"].loc["local", "mean"]
        )
        out["delay_penalty_median_hours"] = float(
            out["lead_time_by_fulfillment"].loc["cross", "median"]
            - out["lead_time_by_fulfillment"].loc["local", "median"]
        )
    else:
        out["delay_penalty_mean_hours"] = np.nan
        out["delay_penalty_median_hours"] = np.nan

    out["cross_rate_by_inv_at_dest"] = (
        mart.groupby("inv_available_at_dc_des")["cross_fulfill"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "cross_rate"})
        .rename(index={0: "no_inv_record", 1: "inv_record"})
    )

    return out

summary = summarize(mart)

print("\n=== Core Metrics ===")
print(f"Cross-fulfillment rate: {summary['cross_fulfillment_rate']:.4f}")
print(f"Lead time (hours) mean / median: {summary['lead_time_mean_hours']:.2f} / {summary['lead_time_median_hours']:.2f}")
print(f"Delay penalty (mean hours): {summary['delay_penalty_mean_hours']:.2f}")
print(f"Delay penalty (median hours): {summary['delay_penalty_median_hours']:.2f}")

print("\n=== Lead Time by Fulfillment Type ===")
print(summary["lead_time_by_fulfillment"])

print("\n=== Late Rate by Fulfillment Type (promise available only) ===")
print(summary["late_rate_by_fulfillment"])

print("\n=== Cross Rate by Inventory Availability at Destination DC ===")
print(summary["cross_rate_by_inv_at_dest"])


=== Core Metrics ===
Cross-fulfillment rate: 0.4776
Lead time (hours) mean / median: 34.66 / 23.86
Delay penalty (mean hours): 17.67
Delay penalty (median hours): 17.35

=== Lead Time by Fulfillment Type ===
                count       mean     median        std
cross_fulfill                                         
local          199851  27.794850  20.444722  25.998340
cross          126983  45.464516  37.797500  31.986325

=== Late Rate by Fulfillment Type (promise available only) ===
                count  late_rate
cross_fulfill                   
local          199662   0.179348
cross          125454   0.117023

=== Cross Rate by Inventory Availability at Destination DC ===
                          count  cross_rate
inv_available_at_dc_des                    
no_inv_record            360899    0.665876
inv_record               189090    0.118303


In [6]:
df = mart[mart["cross_filling_hours"].notna()].copy()

sku_dc_cross_filling_hours = (
    df.groupby(["dc_ori", "dc_des"], as_index=False)
      .agg(
          cross_filling_hours=("cross_filling_hours", "mean"),
          n_orders=("cross_filling_hours", "size")
      )
      .sort_values(["dc_ori", "dc_des", "cross_filling_hours"], ascending=[True, True, True])
)

sku_dc_cross_filling_hours.to_csv('../data/processed/JD_sku_dc_cross_filling_hours.csv', index=False)
print(sku_dc_cross_filling_hours.head(20))

    dc_ori  dc_des  cross_filling_hours  n_orders
0        1       1            13.674569       464
1        1      18            46.000000         1
2        1      31            41.625000         8
3        1      39            36.666667        24
4        1      46            47.000000         1
5        1      53            49.666667         3
6        2       2            13.348242     26539
7        2       6            24.209856      3531
8        2       7            47.859155        71
9        2      15            22.905678      1092
10       2      20            23.753838      3648
11       2      30            22.881384       607
12       2      42            25.392931      2914
13       2      43            26.102061      2038
14       3       3            18.938967       426
15       3      13            25.307692        39
16       3      14            24.582857       175
17       3      16            12.531521      6107
18       3      26            17.046512       774


In [7]:
mart["order_time_dt"] = pd.to_datetime(mart["order_time_dt"], errors="coerce")
mart["order_day"] = mart["order_time_dt"].dt.date

daily_sku_dc = (
    mart
    .groupby(["sku_ID", "dc_des", "order_day"], as_index=False)
    .agg(
        n_orders=("order_ID", "size"),
        cross_rate=("cross_fulfill", "mean"),
        avg_lead_time_hours=("lead_time_hours", "mean"),
        late_delivery_rate=("late_delivery", "mean"),
        inv_available_at_dc_des=("inv_available_at_dc_des", "mean")
    )
)

daily_sku_dc = daily_sku_dc.sort_values(["sku_ID", "dc_des", "order_day"])
daily_sku_dc["inventory"] = (
    daily_sku_dc["n_orders"] * (1 - daily_sku_dc["cross_rate"])
).round().astype(int)

daily_sku_dc.to_csv('../data/processed/JD_daily_sku_dc_summary.csv', index=False)
print(daily_sku_dc.head(20))

        sku_ID  dc_des   order_day  n_orders  cross_rate  avg_lead_time_hours  \
0   000aa92b82       2  2018-03-14         1         0.0                  NaN   
1   000aa92b82       2  2018-03-28         1         0.0                  NaN   
2   000aa92b82       4  2018-03-03         1         0.0                  NaN   
3   000aa92b82       4  2018-03-08         1         0.0                  NaN   
4   000aa92b82       5  2018-03-11         1         0.0                  NaN   
5   000aa92b82       9  2018-03-12         1         0.0                  NaN   
6   000aa92b82       9  2018-03-22         1         0.0                  NaN   
7   000aa92b82      10  2018-03-11         1         0.0                  NaN   
8   000aa92b82      17  2018-03-10         1         1.0                  NaN   
9   000aa92b82      30  2018-03-31         1         1.0                  NaN   
10  000aa92b82      41  2018-03-09         1         1.0                  NaN   
11  000aa92b82      42  2018

In [8]:
dc_day_total = (
    daily_sku_dc
    .groupby(["dc_des", "order_day"], as_index=False)
    .agg(total_orders=("n_orders", "sum"))
)

inventory_capacity = (
    dc_day_total
    .groupby("dc_des", as_index=False)
    .agg(capacity=("total_orders", "max"))
    .sort_values("dc_des")
)

inventory_capacity.to_csv("../data/processed/inventory_capacity.csv", index=False)
print(inventory_capacity.head(10))

   dc_des  capacity
0       1       205
1       2      2949
2       3       192
3       4      3443
4       5      3805
5       6       810
6       7      1130
7       8       103
8       9      3215
9      10       851
